# The "Last Mile" Logistics Auditor
### Client: Veridi Logistics | Analyst: Nadia Iradukunda Hirwa
A delivery performance audit connecting logistics data with customer sentiment to identify where and why Veridi Logistics is failing its customers.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── Imports ───
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("All libraries loaded successfully")

All libraries loaded successfully


## Story 1: The Schema Builder
Loading all raw datasets and performing initial inspection before joining.

In [5]:
# ── File paths ────
BASE_PATH = "/content/"

# ── Load raw CSVs ───
orders      = pd.read_csv(BASE_PATH + "olist_orders_dataset.csv")
reviews     = pd.read_csv(BASE_PATH + "olist_order_reviews_dataset.csv")
customers   = pd.read_csv(BASE_PATH + "olist_customers_dataset.csv")
products    = pd.read_csv(BASE_PATH + "olist_products_dataset.csv")
translation = pd.read_csv(BASE_PATH + "product_category_name_translation.csv")

# ── Quick inspection ───
datasets = {
    "orders": orders,
    "reviews": reviews,
    "customers": customers,
    "products": products,
    "translation": translation
}

for name, df in datasets.items():
    print(f"\n{'─'*50}")
    print(f"📄 {name.upper()}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


──────────────────────────────────────────────────
📄 ORDERS: 99,441 rows × 8 columns
   Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
   Nulls:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

──────────────────────────────────────────────────
📄 REVIEWS: 99,224 rows × 7 columns
   Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
   Nulls:
review_comment_title      87656
review_comment_message    58247
dtype: int64

──────────────────────────────────────────────────
📄 CUSTOMERS: 99,441 rows × 5 columns
   Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
   Nulls:
Series([], dtype: int64)

───────

Joining Orders, Reviews, Customers, and Products into one master dataset.

In [6]:
# Step 1: Keep only DELIVERED orders
orders_delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"Total orders:     {len(orders):,}")
print(f"Delivered orders: {len(orders_delivered):,}")
print(f"Excluded:         {len(orders) - len(orders_delivered):,} (canceled, unavailable, etc.)")

# Step 2: Join Reviews → Orders
# reviews has one review per order (1-to-1), safe to merge
df = orders_delivered.merge(reviews[["order_id", "review_score"]],
                             on="order_id",
                             how="left")
print(f"\nAfter joining reviews: {df.shape}")

# Step 3: Join Customers → Orders
df = df.merge(customers[["customer_id", "customer_state", "customer_city"]],
              on="customer_id",
              how="left")
print(f"After joining customers: {df.shape}")

# Step 4: Check for duplicate rows
dupes = df.duplicated(subset="order_id").sum()
print(f"\n⚠️  Duplicate order_ids: {dupes}")
print("✅ No duplicates!" if dupes == 0 else "❌ Fix needed!")

# Step 5: Preview master dataset
print(f"\nMaster dataset shape: {df.shape}")
df.head(3)

Total orders:     99,441
Delivered orders: 96,478
Excluded:         2,963 (canceled, unavailable, etc.)

After joining reviews: (97007, 9)
After joining customers: (97007, 11)

⚠️  Duplicate order_ids: 529
❌ Fix needed!

Master dataset shape: (97007, 11)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,4.00,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,4.00,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,5.00,GO,vianopolis


In [8]:
# Fix duplicates: some orders have multiple reviews
# Keep the most recent review per order
reviews_deduped = reviews.sort_values("review_answer_timestamp", ascending=False)\
                          .drop_duplicates(subset="order_id", keep="first")

print(f"Reviews before dedup: {len(reviews):,}")
print(f"Reviews after dedup:  {len(reviews_deduped):,}")

#  Rebuild master dataset from scratch
df = orders_delivered.merge(
    reviews_deduped[["order_id", "review_score"]],
    on="order_id",
    how="left"
)
df = df.merge(
    customers[["customer_id", "customer_state", "customer_city"]],
    on="customer_id",
    how="left"
)

# Verify no duplicates
dupes = df.duplicated(subset="order_id").sum()
print(f"\n⚠️  Duplicate order_ids after fix: {dupes}")
print("✅ No duplicates!" if dupes == 0 else "❌ Still broken!")
print(f"Master dataset shape: {df.shape}")

Reviews before dedup: 99,224
Reviews after dedup:  98,673

⚠️  Duplicate order_ids after fix: 0
✅ No duplicates!
Master dataset shape: (96478, 11)


## Story 2: The "Real" Delay Calculator
Calculating actual delivery delays and classifying orders as On Time, Late, or Super Late.

In [9]:
# Convert date columns to datetime
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

# Calculate delay
# Positive = delivered AFTER estimated (late)
# Negative = delivered BEFORE estimated (early)
df["days_difference"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

# Drop rows where we can't calculate delay
before = len(df)
df = df.dropna(subset=["days_difference"])
after = len(df)
print(f"Dropped {before - after:,} rows with missing delivery dates")

# Classify delivery status
def classify_delivery(days):
    if days <= 0:
        return "On Time"
    elif days <= 5:
        return "Late"
    else:
        return "Super Late"

df["delivery_status"] = df["days_difference"].apply(classify_delivery)

# Summary
status_counts = df["delivery_status"].value_counts()
status_pct = df["delivery_status"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Count": status_counts,
    "Percentage": status_pct.round(2)
})
print("\n📊 Delivery Status Summary:")
print(summary)

# Quick chart
colors = {"On Time": "#2ecc71", "Late": "#f39c12", "Super Late": "#e74c3c"}
fig = px.pie(
    values=status_counts.values,
    names=status_counts.index,
    title="Overall Delivery Performance",
    color=status_counts.index,
    color_discrete_map=colors,
    hole=0.4
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

print(f"\n✅ Delay calculation complete. Dataset: {df.shape}")

Dropped 8 rows with missing delivery dates

📊 Delivery Status Summary:
                 Count  Percentage
delivery_status                   
On Time          89936       93.23
Super Late        3764        3.90
Late              2770        2.87



✅ Delay calculation complete. Dataset: (96470, 13)


## Story 3: The Geographic Heatmap
Identifying which Brazilian states have the highest percentage of late deliveries.

In [10]:
# Calculate late delivery % per state
df["is_late"] = df["delivery_status"].isin(["Late", "Super Late"]).astype(int)

state_analysis = df.groupby("customer_state").agg(
    total_orders=("order_id", "count"),
    late_orders=("is_late", "sum"),
    avg_delay_days=("days_difference", "mean"),
    avg_review_score=("review_score", "mean")
).reset_index()

state_analysis["late_pct"] = (
    state_analysis["late_orders"] / state_analysis["total_orders"] * 100
).round(2)

state_analysis = state_analysis.sort_values("late_pct", ascending=False)

print("📊 Top 10 States by Late Delivery %:")
print(state_analysis[["customer_state", "total_orders", "late_pct", "avg_delay_days"]].head(10).to_string(index=False))

📊 Top 10 States by Late Delivery %:
customer_state  total_orders  late_pct  avg_delay_days
            AL           397     21.41           -8.71
            MA           717     17.43           -9.57
            SE           335     15.22          -10.02
            PI           476     13.87          -11.31
            CE          1279     13.76          -10.80
            RR            41     12.20          -17.29
            BA          3256     12.16          -10.79
            RJ         12350     12.11          -11.76
            PA           946     11.21          -14.07
            ES          1995     10.73          -10.50


In [11]:
#  Bar chart: Late delivery % by state
fig = px.bar(
    state_analysis,
    x="customer_state",
    y="late_pct",
    color="late_pct",
    color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"],
    title="🗺️ Late Delivery Rate by State (Late + Super Late)",
    labels={
        "customer_state": "State",
        "late_pct": "Late Delivery Rate (%)"
    },
    text="late_pct"
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    xaxis_tickangle=-45,
    coloraxis_showscale=False,
    height=500,
    plot_bgcolor="white"
)
fig.show()

#  Identify worst regions
worst_states = state_analysis.head(5)
best_states = state_analysis.tail(5)

print("\n🔴 5 Worst States (Highest Late %):")
print(worst_states[["customer_state", "late_pct", "avg_delay_days"]].to_string(index=False))

print("\n🟢 5 Best States (Lowest Late %):")
print(best_states[["customer_state", "late_pct", "avg_delay_days"]].to_string(index=False))


🔴 5 Worst States (Highest Late %):
customer_state  late_pct  avg_delay_days
            AL     21.41           -8.71
            MA     17.43           -9.57
            SE     15.22          -10.02
            PI     13.87          -11.31
            CE     13.76          -10.80

🟢 5 Best States (Lowest Late %):
customer_state  late_pct  avg_delay_days
            PR      4.04          -13.31
            AC      3.75          -20.73
            AP      2.99          -19.69
            RO      2.88          -20.10
            AM      2.76          -19.57


## Story 4: The Sentiment Correlation
Does a late delivery actually cause a bad review? Connecting logistics performance to customer satisfaction.

In [12]:
# Average review score by delivery status
sentiment_summary = df.groupby("delivery_status").agg(
    avg_review_score=("review_score", "mean"),
    order_count=("order_id", "count")
).reset_index()

sentiment_summary["avg_review_score"] = sentiment_summary["avg_review_score"].round(2)
print("📊 Average Review Score by Delivery Status:")
print(sentiment_summary.to_string(index=False))

# Bar chart: Review score by delivery status
colors = {"On Time": "#2ecc71", "Late": "#f39c12", "Super Late": "#e74c3c"}
fig = px.bar(
    sentiment_summary,
    x="delivery_status",
    y="avg_review_score",
    color="delivery_status",
    color_discrete_map=colors,
    title="Average Review Score by Delivery Status",
    labels={
        "delivery_status": "Delivery Status",
        "avg_review_score": "Average Review Score (1–5)"
    },
    text="avg_review_score"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    yaxis_range=[0, 5.5],
    plot_bgcolor="white",
    height=450
)
fig.show()

# ── Scatter: Delay days vs review score ──────────────────
# Sample 5000 points so chart isn't too heavy
df_sample = df.sample(5000, random_state=42)

fig2 = px.scatter(
    df_sample,
    x="days_difference",
    y="review_score",
    color="delivery_status",
    color_discrete_map=colors,
    title="Delivery Delay (Days) vs Review Score",
    labels={
        "days_difference": "Days Difference (Negative = Early)",
        "review_score": "Review Score (1–5)"
    },
    opacity=0.4,
    trendline="ols"
)
fig2.update_layout(plot_bgcolor="white", height=450)
fig2.show()

print("\n✅ Sentiment analysis complete")

📊 Average Review Score by Delivery Status:
delivery_status  avg_review_score  order_count
           Late              2.99         2770
        On Time              4.29        89936
     Super Late              1.74         3764



✅ Sentiment analysis complete


## Bonus Story: The "Translation" Challenge
Translating Portuguese product categories to English for business readability.

In [13]:
# Join products with translation
products_translated = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# Fill untranslated categories
products_translated["product_category_name_english"] = (
    products_translated["product_category_name_english"]
    .fillna("Unknown")
)

print(f"Products with English category: {(products_translated['product_category_name_english'] != 'Unknown').sum():,}")
print(f"Products without translation:   {(products_translated['product_category_name_english'] == 'Unknown').sum():,}")
print(f"\nSample translations:")
print(products_translated[["product_category_name", "product_category_name_english"]].drop_duplicates().head(10).to_string(index=False))

Products with English category: 32,328
Products without translation:   623

Sample translations:
product_category_name product_category_name_english
           perfumaria                     perfumery
                artes                           art
        esporte_lazer                sports_leisure
                bebes                          baby
utilidades_domesticas                    housewares
instrumentos_musicais           musical_instruments
           cool_stuff                    cool_stuff
     moveis_decoracao               furniture_decor
     eletrodomesticos               home_appliances
           brinquedos                          toys


## Candidate's Choice: Revenue-at-Risk Analysis
**Why this matters:** Knowing *where* deliveries are late is useful, but the CEO needs to know *how much it costs*. By combining late delivery rates with order volume per state, we can calculate which states represent the biggest business risk — so the repair budget can be allocated where it has maximum impact.

In [18]:
# ── Candidate's Choice: High-Risk State Prioritization ───
# Risk Score = late_order_volume (absolute number of late orders)
# This is more honest: RJ with 1,495 late orders hurts more than AL with 85

state_analysis["risk_score"] = (
    state_analysis["late_pct"] * np.log1p(state_analysis["total_orders"])
).round(2)

state_analysis["late_order_volume"] = state_analysis["late_orders"].astype(int)

# Separate rankings
by_late_pct    = state_analysis.sort_values("late_pct", ascending=False)
by_late_volume = state_analysis.sort_values("late_order_volume", ascending=False)
by_risk_score  = state_analysis.sort_values("risk_score", ascending=False)

state_risk = by_risk_score.copy()

print("🎯 Top 10 States by LATE ORDER VOLUME (absolute business impact):")
print(by_late_volume[["customer_state", "total_orders", "late_pct",
                       "late_order_volume"]].head(10).to_string(index=False))

print("\n🎯 Top 10 States by RISK SCORE (volume × rate):")
print(by_risk_score[["customer_state", "total_orders", "late_pct",
                      "late_order_volume", "risk_score"]].head(10).to_string(index=False))

# ── Bubble chart ──────────────────────────────────────────
fig = px.scatter(
    state_analysis,
    x="total_orders",
    y="late_pct",
    size="late_order_volume",
    color="risk_score",
    text="customer_state",
    color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"],
    title="🎯 Business Risk by State: Order Volume vs Late Delivery Rate",
    labels={
        "total_orders": "Total Orders (Volume)",
        "late_pct": "Late Delivery Rate (%)",
        "risk_score": "Risk Score"
    },
    size_max=60
)
fig.update_traces(textposition="top center")
fig.update_layout(plot_bgcolor="white", height=550)
fig.show()

🎯 Top 10 States by LATE ORDER VOLUME (absolute business impact):
customer_state  total_orders  late_pct  late_order_volume
            SP         40494      4.49               1820
            RJ         12350     12.11               1495
            MG         11354      4.57                519
            BA          3256     12.16                396
            RS          5344      6.08                325
            SC          3546      8.21                291
            ES          1995     10.73                214
            PR          4923      4.04                199
            CE          1279     13.76                176
            PE          1593      9.60                153

🎯 Top 10 States by RISK SCORE (volume × rate):
customer_state  total_orders  late_pct  late_order_volume  risk_score
            AL           397     21.41                 85      128.17
            MA           717     17.43                125      114.63
            RJ         12350     12.11 

## Story 4 Extended: Key Insights Summary

In [19]:
# ── Key numbers ───────────────────────────────────────────
on_time_score    = sentiment_summary.loc[sentiment_summary["delivery_status"] == "On Time",     "avg_review_score"].values[0]
late_score       = sentiment_summary.loc[sentiment_summary["delivery_status"] == "Late",        "avg_review_score"].values[0]
super_late_score = sentiment_summary.loc[sentiment_summary["delivery_status"] == "Super Late",  "avg_review_score"].values[0]

total_late       = state_analysis["late_orders"].sum()
total_orders_all = state_analysis["total_orders"].sum()
overall_late_pct = round(total_late / total_orders_all * 100, 2)

# Highest late RATE state
worst_rate_state     = by_late_pct.iloc[0]["customer_state"]
worst_rate_pct       = by_late_pct.iloc[0]["late_pct"]

# Highest late VOLUME state (most orders affected)
worst_volume_state   = by_late_volume.iloc[0]["customer_state"]
worst_volume_count   = int(by_late_volume.iloc[0]["late_order_volume"])
worst_volume_orders  = int(by_late_volume.iloc[0]["total_orders"])

print("=" * 60)
print("📋 EXECUTIVE FINDINGS — VERIDI LOGISTICS AUDIT")
print("=" * 60)
print(f"""
🚚 DELIVERY PERFORMANCE
   • {overall_late_pct}% of delivered orders arrived late
   • On Time orders average:    {on_time_score} ⭐
   • Late orders average:       {late_score} ⭐
   • Super Late orders average: {super_late_score} ⭐

🗺️  GEOGRAPHIC FINDINGS
   • Highest late RATE:    {worst_rate_state} ({worst_rate_pct}% of orders late)
   • Highest late VOLUME:  {worst_volume_state} ({worst_volume_count:,} late orders
                           out of {worst_volume_orders:,} total)
   • Remote northern states (AM, RO, AP) have surprisingly
     LOW late rates despite geographic distance

💡 RECOMMENDATION
   The problem is NOT nationwide — it is concentrated in
   northeastern states (AL, MA, SE, PI, CE).

   Two-pronged fix:
   → Fix {worst_rate_state} first (worst late rate at {worst_rate_pct}%)
   → Fix {worst_volume_state} urgently (most customers affected:
     {worst_volume_count:,} late deliveries)
""")
print("=" * 60)

📋 EXECUTIVE FINDINGS — VERIDI LOGISTICS AUDIT

🚚 DELIVERY PERFORMANCE
   • 6.77% of delivered orders arrived late
   • On Time orders average:    4.29 ⭐
   • Late orders average:       2.99 ⭐
   • Super Late orders average: 1.74 ⭐

🗺️  GEOGRAPHIC FINDINGS
   • Highest late RATE:    AL (21.41% of orders late)
   • Highest late VOLUME:  SP (1,820 late orders
                           out of 40,494 total)
   • Remote northern states (AM, RO, AP) have surprisingly
     LOW late rates despite geographic distance

💡 RECOMMENDATION
   The problem is NOT nationwide — it is concentrated in
   northeastern states (AL, MA, SE, PI, CE).

   Two-pronged fix:
   → Fix AL first (worst late rate at 21.41%)
   → Fix SP urgently (most customers affected:
     1,820 late deliveries)

